# Session 14 · Exercises

Work in pairs. The first run of the setup cells downloads the tokeniser and the embedding model `intfloat/multilingual-e5-small` (about 470 MB). The lecturer chooses which exercises to set; you will not have time for all of them. Run the starter cells (marked `# starter`) and write your answers in the empty cells.

Part 2 has one exercise that needs a language model: a local Ollama server or an OpenAI-compatible API (see the session README).

*Author: course team, licence CC BY 4.0.*

In [ ]:
# starter: load the case study (works wherever Jupyter was started inside the repository)
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
year = decisions["start_date"].dt.year
print(decisions.shape)

## Part 1 · Embeddings: from words to vectors, and what they are good for

In [ ]:
# starter: the embedding model and its tokeniser (loaded from the local cache)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")
tokenizer = model.tokenizer                 # SentencePiece tokeniser of the model
print(model.max_seq_length, "tokens input limit")

### 1.1 · What is tezgüino?

*Paper, 5 min.*

> "A bottle of tezgüino is on the table. Everyone likes tezgüino. Tezgüino makes you drunk. We make tezgüino out of corn." What is it, and which words told you?

*Source:* [Lin (1998), "Automatic retrieval and clustering of similar words" (example after Nida 1975)](https://aclanthology.org/P98-2127/)

*Your answer:*

### 1.2 · Cosine similarity by hand

*Paper, 10 min.*

> Three two-dimensional "embeddings": which pair is closest?
>
> a = (2, 1) "Spielzeugauto", b = (3, 2) "voiture jouet", c = (1, 3) "Damenschuhe".
>
> Compute the three cosine similarities. Then add d = (4, 2): what is cos(a, d), and what is the Euclidean distance between a and d? What does this say about the length of an embedding vector?

*Source:* [Jurafsky & Martin, *Speech and Language Processing*, Chapter 5: cosine similarity](https://web.stanford.edu/~jurafsky/slp3/5.pdf)

*Your answer:*

### 1.3 · Tokens per language

*Predict, then Code, 10 min.*

> One product description in English, German, Hungarian and Greek (in the starter cell below). Predict the token counts, then count them with the local tokenizer.
>
> Write your four predictions down before you run anything. Then count words and tokens (`tokenizer.tokenize`) and compute tokens per word for each language. Which language would cost the most in an API that bills per token?

*Source:* [Petrov et al. (2023), "Language model tokenizers introduce unfairness between languages"](https://arxiv.org/abs/2305.15425)

In [ ]:
# starter: the same toy car in four languages (translations by the course team)
texts = {
    "en": "Toy car for children, made of plastic, with four wheels and a battery-powered electric motor.",
    "de": "Spielzeugauto für Kinder aus Kunststoff, mit vier Rädern und einem batteriebetriebenen Elektromotor.",
    "hu": "Gyermekeknek szánt műanyag játékautó négy kerékkel és elemmel működő elektromos motorral.",
    "el": "Παιχνίδι αυτοκίνητο για παιδιά, από πλαστικό, με τέσσερις τροχούς και ηλεκτρικό κινητήρα που λειτουργεί με μπαταρίες.",
}

*Your predictions (before running code):*

### 1.4 · Swap the model name

*Code, 10 min.*

> Notebook 02 showed that every model comes with its own tokeniser and that `AutoTokenizer.from_pretrained(name)` loads it. Tokenise the German and the French description below with the tokenisers of three models: `intfloat/multilingual-e5-small`, `google-bert/bert-base-german-cased` and `distilbert/distilbert-base-uncased-finetuned-sst-2-english`. Change only the name. Count the tokens and look at the pieces: which tokeniser splits the French text into the most pieces, which one loses the capital letters, and why? Which of the three models would you use for descriptions in 22 languages?

In [ ]:
# starter: two descriptions of women's leather shoes
descriptions = {
    "de": "Damenstiefel mit Oberteil aus Rindleder und Laufsohle aus Gummi",
    "fr": "Chaussures pour femmes, dessus en cuir, semelle en caoutchouc",
}
names = ["intfloat/multilingual-e5-small",
         "google-bert/bert-base-german-cased",
         "distilbert/distilbert-base-uncased-finetuned-sst-2-english"]

### 1.5 · Same goods, different languages

*Code, 10 min.*

> Take the decisions of two headings, 9503 (toys) and 6403 (footwear with leather uppers), and draw two decisions per heading in each of German, French and Czech (12 decisions, `random_state=0`). Embed them with the `"passage: "` prefix and compute the 12 × 12 similarity matrix. Compare the mean similarity of pairs with the same heading but a different language with that of pairs with the same language but a different heading. Read the English `keywords` of the decisions to check what they are about. Which matters more for the model: the goods or the language?

### 1.6 · Search the French decisions

*Pairs, 15 min.*

> The starter cell below loads 10,000 training decisions and their embeddings. Build a search over the **French** decisions only, with embeddings and with TF-IDF. Search with two German queries and one English query of your own (for example a product you own), and judge the top 5 of each method with the English `keywords`. Then find one query for which TF-IDF gives the more useful top 5 and explain why.

In [ ]:
# starter: 10,000 training decisions and their e5 vectors (shared cache with case study 04;
# about a minute to encode if the cache file does not exist yet)
train = decisions[year <= 2021].sample(10_000, random_state=0)
cache = Path("embeddings_cache")
cache.mkdir(exist_ok=True)
path = cache / "multilingual-e5-small_passage_train.npy"
if path.exists():
    E_tr = np.load(path)
else:
    E_tr = model.encode(("passage: " + train["description"]).tolist(), batch_size=64,
                        normalize_embeddings=True, show_progress_bar=False)
    np.save(path, E_tr)
french = (train["language"] == "fr").to_numpy()
print(E_tr.shape, french.sum(), "French decisions")

### 1.7 · *Stretch* · Clusters of toys

*Code, 15 min.*

> Cluster the training decisions of chapter 95 (toys, games and sports equipment; `train["chapter"] == "95"`) with k-means on their embeddings, for k = 4 and k = 8. Cross-tabulate the clusters against `heading` and against `language`, and compute the adjusted Rand index with the headings. Do the clusters reflect goods or languages?

## Part 2 · Generative models through an API

### 2.1 · Be the language model

*Class, 10 min.*

> Each student adds the most likely next word to a sentence. Repeat, rolling a die to choose among the three most likely words.

*Source:* [Shannon (1951), "Prediction and entropy of printed English"](https://www.princeton.edu/~wbialek/rome/refs/shannon_51.pdf) (Shannon's subjects guessed letters); temperature: [Jurafsky & Martin, Chapter 7](https://web.stanford.edu/~jurafsky/slp3/7.pdf)

*Your answer:*

### 2.2 · Bill the class

*Paper, 5 min.*

> 113,000 decisions × 300 tokens at a given price per million tokens. Compare with the cost of TF-IDF.
>
> Use the example prices of lecture notebook 05: 0.15 USD per million input tokens and 0.60 USD per million output tokens, and 40 output tokens per decision for the JSON answer (the lecturer may give you a current price list instead; note its date). What does one run over the test set cost? What does it cost if every prompt also contains the English texts of all 1,229 headings (33,000 tokens)?

*Source:* original exercise for this course; background: [Claude API pricing](https://platform.claude.com/docs/en/about-claude/pricing) (prices change)

*Your answer:*

### 2.3 · Write the ambiguous case

*Pairs, 10 min.*

> Write a product description that a zero-shot prompt will misclassify. Swap and explain each other's trap.
>
> If a local model is running, try your description with the `classify` function of lecture notebook 05.

*Source:* original exercise for this course; background: [Ribeiro et al. (2020), "Beyond accuracy: behavioral testing of NLP models with CheckList"](https://aclanthology.org/2020.acl-main.442/)

*Your answer:*

### 2.4 · Abstention at another price of an error

*Code, 15 min.*

> The lecture chose the threshold *t* for an officer's check costing 1 unit and an accepted wrong heading costing 5. Repeat the choice for an error costing 2 units and for an error costing 10 units: choose *t* on the 2022 decisions, then report coverage, accuracy on the accepted requests and cost per request on 2023. Then show, for your two thresholds, the share of requests the model decides on its own per language (German, French, English, other). Which requests would an officer see most of?

In [ ]:
# starter: the TF-IDF classifier with probabilities from lecture notebook 05 (about 15 seconds)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline

train_all, valid = decisions[year <= 2021], decisions[year >= 2022]
prob_clf = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True),
                         SGDClassifier(loss="log_loss", alpha=1e-6, random_state=0, n_jobs=-1))
prob_clf.fit(train_all["description"], train_all["heading"])
proba = prob_clf.predict_proba(valid["description"])
confidence = proba.max(axis=1)                                         # the model's confidence
correct = prob_clf.classes_[proba.argmax(axis=1)] == valid["heading"].to_numpy()
in_2022 = (valid["start_date"].dt.year == 2022).to_numpy()
print(len(valid), round(correct.mean(), 3))

### 2.5 · A recommendation for the head of unit

*Pairs, 10 min.*

> Case study 06 measured, on the same 200 decisions from 2022–2023: TF-IDF classifier accuracy 0.75 (95 % bootstrap interval 0.69 to 0.81), under 1 ms per decision, no data leave the machine; the true heading is among the classifier's ten candidates for 88 % of them. Lecture notebook 05 estimates 0.5–5 seconds per decision for a language model; a hosted model sends each request to the provider, a local one keeps it on the machine. Write a recommendation of five sentences for the head of a customs classification unit: which method would you deploy as a suggestion tool for officers, and why? Address quality, cost, latency and data protection (new BTI requests contain confidential business information), and say what you would measure before deciding.

*Your answer:*

### 2.6 · *Stretch* · A local model on footwear

*Code, 15 min. Needs the local model (Ollama with `qwen2.5:3b`, see the session README).*

> The starter cell draws 20 decisions of chapter 64 (footwear) from 2022–2023. Let the local model choose zero-shot among the six headings 6401–6406, with their English texts in the prompt and structured output, at temperature 0. Compare its accuracy with the TF-IDF classifier of exercise 2.4 on the same 20 decisions, and record the seconds per decision. Read the decisions on which they disagree.

In [ ]:
# starter: model server and the 20 footwear decisions
import os

import httpx
from openai import OpenAI

BASE_URL = os.getenv("LLM_BASE_URL", "http://localhost:11434/v1")
MODEL = os.getenv("LLM_MODEL", "qwen2.5:3b")
API_KEY = os.getenv("LLM_API_KEY", "ollama")
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60)
try:
    LLM_AVAILABLE = httpx.get(BASE_URL.rstrip("/") + "/models",
                              headers={"Authorization": f"Bearer {API_KEY}"}, timeout=3).status_code == 200
except httpx.HTTPError:
    LLM_AVAILABLE = False
print(BASE_URL, MODEL, "server reachable:", LLM_AVAILABLE)

footwear = valid[valid["chapter"] == "64"].sample(20, random_state=0)
candidates = ["6401", "6402", "6403", "6404", "6405", "6406"]
print(footwear["heading"].value_counts().to_dict())
print(nomenclature.loc[candidates, "heading_description"].str[:80].to_dict())